<a href="https://colab.research.google.com/github/ewarren38/HW7_WarrenE/blob/main/HW7_WarrenE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Read in the red and white wine quality data, and display the first few values. The delimiter here is a `;`.

In [74]:
import pandas as pd
red = pd.read_csv("/content/winequality-red.csv", sep = ";")
red["type"] = "red"
red.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red


In [75]:
white = pd.read_csv("/content/winequality-white.csv", sep = ";")
white["type"] = "white"
white.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6,white
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6,white
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6,white
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6,white
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6,white


In [76]:
wine = pd.concat([red, white], ignore_index = True)
wine

# there are no missing values in the data
#wine.isna().count()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6492,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6,white
6493,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5,white
6494,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6,white
6495,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7,white


In [77]:
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, LassoCV, Lasso
from math import sqrt
import numpy as np
#import statsmodels.api as sm

First, for our MLR model, we'll want to treat `alcohol` as our response. Because of that, we'll exclude it from our training data. We'll also use `type` as a stratification variable, since we want to ensure that we have a similar proportion of red and white wines in the training and test sets.

In [78]:
X_train, X_test, y_train, y_test = train_test_split(
  wine.drop(["alcohol", "quality"], axis = 1), # axis = 1 means you're dropping a column.
  wine["alcohol"], # use alcohol as our response
  test_size=0.20, # 80 / 20 split
  random_state=41, # setting a seed
  stratify = wine["type"]) # our stratify column

Before we make our MLR models, we will want to standardize our regressors, since the sulfur metrics are much larger in magnitude than the other variables and we don't want that to artifically cause it to matter more in our model.

We don't want to standardize our **test** set by it's own mean/std, we want to standardize it by our **training** set mean/std. That's because when we're assessing our model we don't want anything to rely on the **test** set!

In [79]:
# exclude "type" since we can't find the mean/std of a string
X_train_num = X_train.drop("type", axis = 1)
means = X_train_num.apply(np.mean, axis = 0)
means

,0
fixed acidity,7.219925
volatile acidity,0.339677
citric acid,0.318339
residual sugar,5.472321
chlorides,0.056219
free sulfur dioxide,30.419473
total sulfur dioxide,115.381278
density,0.994713
pH,3.218232
sulphates,0.531339


In [80]:
std = X_train_num.apply(np.std, axis = 0)
std

,0
fixed acidity,1.295153
volatile acidity,0.165221
citric acid,0.144098
residual sugar,4.769460
chlorides,0.035877
free sulfur dioxide,17.715183
total sulfur dioxide,55.981301
density,0.003021
pH,0.160119
sulphates,0.150413


In [81]:
# For all the variables in our x training dataset, replace the variable with itself
# minus (elementwise) the mean of itself, divided by the standard deviation of itself
# which we specify by axis = 0 so that these calculations are done up and down the rows
# of a variable.
X_train_standardized = X_train_num.apply(lambda x: (x-np.mean(x))/np.std(x), axis = 0) # axis = 0 means we take in columns
X_train_standardized["type"] = X_train["type"] # put "type" back in

# Replace our training dataset with the standardized version that includes type
X_train = X_train_standard
X_train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,type,den_sulph_int,chlor_sulph_int,ph_square
4185,-0.324228,-0.966444,-0.127270,1.536375,0.161139,-0.644615,0.993523,1.114684,-1.362938,-0.075387,white,-0.084033,-0.012148,1.857600
3599,-0.324228,-0.179618,-1.584610,-0.853833,-0.452065,1.443989,0.529082,-0.831770,-0.176317,-0.474290,white,0.394500,0.214410,0.031088
4631,-0.401439,-1.208544,0.983084,-0.811899,-0.563556,-0.870410,-0.417662,-0.686117,0.947850,-0.274839,white,0.188571,0.154887,0.898419
4009,-0.169806,-1.087494,-0.404859,0.383205,-0.256954,-0.023679,1.868815,0.498969,1.010303,0.323515,white,0.161424,-0.083129,1.020713
2794,0.525093,-0.724344,0.913687,1.829910,-0.507811,1.274643,0.457630,1.485437,-1.175577,-0.474290,white,-0.704528,0.240849,1.381981
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5899,-0.633072,-0.300669,-0.127270,0.215471,4.258459,1.782681,0.725577,0.022286,-0.738401,-0.873192,white,-0.019460,-3.718454,0.545236
1913,-1.096338,0.001957,-0.751845,0.236438,-0.452065,1.105296,0.922071,-0.202814,0.448220,0.589450,white,-0.119549,-0.266470,0.200901
1526,-0.324228,0.788782,-1.654007,-0.686099,0.216885,-0.701064,-1.382270,0.270559,0.510674,0.788901,red,0.213444,0.171101,0.260788
1975,-0.633072,-0.179618,-0.266065,-0.832866,-0.535684,-1.039756,0.064642,-0.931079,0.635581,-0.141871,white,0.132093,0.075998,0.403963


Creating a version of our standardized training data that includes some interaction terms.

- `den_sulph_int`: is the interaction between density and sulphates

- `chlor_sulph_int`: is the interaction between chlorides and sulphates

In [82]:
X_train["chlor_sulph_int"] = X_train["chlorides"]*X_train["sulphates"]
X_train["pH_square"] = X_train["pH"]**2
X_train = pd.get_dummies(X_train, columns = ["type"])
X_train.drop("type_white", axis = 1)
X_train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,den_sulph_int,chlor_sulph_int,ph_square,pH_square,type_red,type_white
4185,-0.324228,-0.966444,-0.127270,1.536375,0.161139,-0.644615,0.993523,1.114684,-1.362938,-0.075387,-0.084033,-0.012148,1.857600,1.857600,False,True
3599,-0.324228,-0.179618,-1.584610,-0.853833,-0.452065,1.443989,0.529082,-0.831770,-0.176317,-0.474290,0.394500,0.214410,0.031088,0.031088,False,True
4631,-0.401439,-1.208544,0.983084,-0.811899,-0.563556,-0.870410,-0.417662,-0.686117,0.947850,-0.274839,0.188571,0.154887,0.898419,0.898419,False,True
4009,-0.169806,-1.087494,-0.404859,0.383205,-0.256954,-0.023679,1.868815,0.498969,1.010303,0.323515,0.161424,-0.083129,1.020713,1.020713,False,True
2794,0.525093,-0.724344,0.913687,1.829910,-0.507811,1.274643,0.457630,1.485437,-1.175577,-0.474290,-0.704528,0.240849,1.381981,1.381981,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5899,-0.633072,-0.300669,-0.127270,0.215471,4.258459,1.782681,0.725577,0.022286,-0.738401,-0.873192,-0.019460,-3.718454,0.545236,0.545236,False,True
1913,-1.096338,0.001957,-0.751845,0.236438,-0.452065,1.105296,0.922071,-0.202814,0.448220,0.589450,-0.119549,-0.266470,0.200901,0.200901,False,True
1526,-0.324228,0.788782,-1.654007,-0.686099,0.216885,-0.701064,-1.382270,0.270559,0.510674,0.788901,0.213444,0.171101,0.260788,0.260788,True,False
1975,-0.633072,-0.179618,-0.266065,-0.832866,-0.535684,-1.039756,0.064642,-0.931079,0.635581,-0.141871,0.132093,0.075998,0.403963,0.403963,False,True


In [88]:
# Model with all of the original predictors in it (plus type)
cv_mlr1 = cross_validate(
    LinearRegression(),
    X_train[["fixed acidity", "volatile acidity", "citric acid", \
             "residual sugar", "chlorides", "free sulfur dioxide", \
             "total sulfur dioxide", "density", "pH", "sulphates", "type_red"]],
    y_train,
    cv = 5, # 5 folds cross validation
    scoring = "neg_mean_squared_error") # MSE is our metric for how good it is

# Model with a selection of what I think are important
cv_mlr2= cross_validate(
    LinearRegression(),
    X_train[["fixed acidity", "residual sugar", "chlorides", \
             "sulphates", "pH", "type_red"]],
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

# Model with a polynomial term for pH
cv_mlr3 = cross_validate(
    LinearRegression(),
    X_train[["fixed acidity", "residual sugar", "chlorides", \
             "sulphates", "pH", "pH_square", "type_red"]], # these variables in our model
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

# Model with an interaction term for chlorides and sulfates
cv_mlr4 = cross_validate(
    LinearRegression(),
    X_train[["fixed acidity", "residual sugar", "chlorides", \
             "sulphates", "chlor_sulph_int", "pH", "type_red"]], # these variables in our model
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")

Find the root MSE for each of the MLR models. We will choose the model with the smallest value.

In [89]:
print(np.sqrt(-sum(cv_mlr1['test_score'])),
      np.sqrt(-sum(cv_mlr2['test_score'])),
      np.sqrt(-sum(cv_mlr3['test_score'])),
      np.sqrt(-sum(cv_mlr4['test_score'])))

1.1671901816456545 2.341350569031405 2.340955854417562 2.312428979857177


Fit our best model on all of our training data now (not just each individual fold)

In [93]:
mlr_best = LinearRegression().fit(X_train[["fixed acidity", "volatile acidity", \
                                           "citric acid", "residual sugar", \
                                           "chlorides", "free sulfur dioxide", \
                                           "total sulfur dioxide", "density", \
                                           "pH", "sulphates", "type_red"]], y_train)
print(mlr_best.intercept_)
print(np.array(list(zip(X_train.columns, mlr_best.coef_))))

10.198955286519958
[['fixed acidity' '0.6902503833630642']
 ['volatile acidity' '0.11284883225583897']
 ['citric acid' '0.07073893857116548']
 ['residual sugar' '1.1454414151509014']
 ['chlorides' '-0.037318205464376086']
 ['free sulfur dioxide' '-0.05427581245990906']
 ['total sulfur dioxide' '-0.02822429884654154']
 ['density' '-2.04342054239308']
 ['pH' '0.43416777135222084']
 ['sulphates' '0.16202301223946836']
 ['den_sulph_int' '1.179465240517699']]


Next, do a LASSO model with at least 5 predictors

In [94]:
# we aren't specifying our alphas, we're going to let LASSO find it for us?
# 5 cross-validation folds
lasso_mod = LassoCV(cv=5, random_state=0) \
    .fit(X_train[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", \
                  "total sulfur dioxide", "pH", \
                  "sulphates", "type_red"]],
         y_train)

In [95]:
# The number of folds and variables does not determine the number of automatic alphas
# python produces. It'll always spit out 100.
lasso_mod.alphas_

array([0.43358481, 0.40436265, 0.37710996, 0.35169401, 0.32799101,
       0.30588551, 0.28526985, 0.26604362, 0.24811317, 0.23139117,
       0.21579618, 0.20125224, 0.18768851, 0.17503894, 0.1632419 ,
       0.15223994, 0.14197948, 0.13241054, 0.12348651, 0.11516393,
       0.10740227, 0.10016372, 0.09341302, 0.08711729, 0.08124588,
       0.07577018, 0.07066352, 0.06590104, 0.06145953, 0.05731736,
       0.05345436, 0.04985172, 0.04649188, 0.04335848, 0.04043626,
       0.037711  , 0.0351694 , 0.0327991 , 0.03058855, 0.02852699,
       0.02660436, 0.02481132, 0.02313912, 0.02157962, 0.02012522,
       0.01876885, 0.01750389, 0.01632419, 0.01522399, 0.01419795,
       0.01324105, 0.01234865, 0.01151639, 0.01074023, 0.01001637,
       0.0093413 , 0.00871173, 0.00812459, 0.00757702, 0.00706635,
       0.0065901 , 0.00614595, 0.00573174, 0.00534544, 0.00498517,
       0.00464919, 0.00433585, 0.00404363, 0.0037711 , 0.00351694,
       0.00327991, 0.00305886, 0.0028527 , 0.00266044, 0.00248

In [96]:
# take the array of 100 alphas and the corresponding MSE's and zip them together
# across columns, then make a list of those two-pair lists and print in descending
# order according to MSE to figure out which alpha yields the lowest MSE.

np.set_printoptions(suppress = True)
fit_info = np.array(list(zip(lasso_mod.alphas_, np.mean(lasso_mod.mse_path_, axis = 1))))
fit_info.shape # there are 100 rows, and 2 columns

# this means that across all 100 rows (of dim1), we
# look at the second column (index 1) of dim2, and sort all the arguments
# and that gives us the indexes in sorted order?
fit_info[:,1].argsort()

# lastly, we want to print off the first 10 rows, across all columns, rounded to 4
# of our array in the index order specified from the sort.
fit_info[fit_info[:,1].argsort()][0:10,].round(4)

array([[0.0004, 0.9857],
       [0.0005, 0.9857],
       [0.0005, 0.9857],
       [0.0005, 0.9857],
       [0.0006, 0.9857],
       [0.0006, 0.9857],
       [0.0007, 0.9857],
       [0.0007, 0.9857],
       [0.0008, 0.9857],
       [0.0008, 0.9857]])

Now we have the alpha, intercept, and coefficients for our best LASSO model. Notice that none of the variables were set equal to 0, which could've happened since LASSO helps us do model selection. That's a good sign that we chose informative columns.

In [97]:
print(lasso_mod.alpha_)
print(lasso_mod.intercept_)
print(np.array(list(zip(X_train.columns, lasso_mod.coef_))))

0.0004335848127892822
10.658794179227609
[['fixed acidity' '-0.1353685258194693']
 ['volatile acidity' '0.13048029585406254']
 ['citric acid' '-0.349637713563351']
 ['residual sugar' '-0.3348425488458766']
 ['chlorides' '-0.4884273672802044']
 ['free sulfur dioxide' '0.03198783577901725']
 ['total sulfur dioxide' '0.09134847707303433']
 ['density' '-0.6890122617507567']]
